# Feature Engineering

In [1]:
import holidays
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OrdinalEncoder

from src.utils.helpers import get_best_k
from src.plots import plot_scatter, plot_correlation_matrix

pd.set_option('display.max_columns', None)

ModuleNotFoundError: No module named 'src'

In [47]:
#importando dados
df = pd.read_pickle('../data/curated/data.pkl')

**Obtendo atraso no voo anterior da aeronave**

Esta variável aponta a dependência sequencial da malha aérea ao identificar se a aeronave estava em atraso no voo anterior, permitindo que o modelo compreenda o efeito cascata observado na análise exploratória. Para garantir que o modelo capture apenas a inércia operacional real, refinamos a métrica excluindo casos com janelas de recuperação superiores a 6 horas. Esse ajuste busca eliminar o ruído de atrasos já absorvidos pela logística, permitindo que o algoritmo foque estritamente nos gargalos sistêmicos que se acumulam ao longo do tempo e degradam a pontualidade.

In [48]:
df['SCHEDULED_DEPARTURE_DATETIME'] = pd.to_datetime(df[['YEAR', 'MONTH', 'DAY']].assign(
    hour=df['SCHEDULED_DEPARTURE'] // 100, 
    minute=df['SCHEDULED_DEPARTURE'] % 100
))

df = df.sort_values(by=['TAIL_NUMBER', 'SCHEDULED_DEPARTURE_DATETIME'])

df['AIRPLANE_WAS_DELAYED'] = df.groupby('TAIL_NUMBER')['IS_DELAYED'].shift(1).fillna(0).astype(int)

#1 se a diferença for > 6 horas, 0 se for <= 6 horas
df['AIRPLANE_TIME_BETWEEN_FLIGHTS'] = df.groupby('TAIL_NUMBER')['SCHEDULED_DEPARTURE_DATETIME'].diff().dt.total_seconds() / 3600
df['HAS_RECOVERY_WINDOW'] = (df['AIRPLANE_TIME_BETWEEN_FLIGHTS'] > 6).astype(int)

df['AIRPLANE_WAS_DELAYED'] = ((df['AIRPLANE_WAS_DELAYED'] == 1) & (df['HAS_RECOVERY_WINDOW'] == 0)).astype(int)

df[['TAIL_NUMBER', 'AIRPLANE_WAS_DELAYED']].head(20)

,TAIL_NUMBER,AIRPLANE_WAS_DELAYED
3052236,7819A,0
3055380,7819A,0
3059072,7819A,0
3061230,7819A,0
3065979,7819A,0
3066890,7819A,1
3068892,7819A,1
3071094,7819A,0
3073040,7819A,0
3074872,7819A,0


**Obtendo taxa de atraso no aeroporto de origem na última hora**

Esta variável atua como um sensor de degradação operacional ao capturar a volatilidade do aeroporto de origem no período imediatamente anterior à decolagem. Ela funciona como uma variável para fatores externos não estruturados, como meteorologia adversa ou saturação do controle de tráfego, permitindo que o modelo identifique 'ondas' de atraso. Ao mapear essa autocorrelação temporal, o algoritmo ajusta o risco com base no estado atual da malha, mitigando a ausência de bases de dados externas.

In [49]:
df = df.sort_values(['ORIGIN_AIRPORT', 'SCHEDULED_DEPARTURE_DATETIME']).reset_index(drop=True)

df['ORIGIN_AIRPORT_DELAY_MOMENTUM'] = (
    df.groupby('ORIGIN_AIRPORT')
    .rolling('1h', on='SCHEDULED_DEPARTURE_DATETIME', closed='left')['IS_DELAYED']
    .mean()
    .reset_index(level=0, drop=True)
    .fillna(0).values
)

df[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_DELAY_MOMENTUM']].head(20)

,ORIGIN_AIRPORT,ORIGIN_AIRPORT_DELAY_MOMENTUM
0,ABE,0.0
1,ABE,0.0
2,ABE,0.0
3,ABE,0.0
4,ABE,0.0
5,ABE,0.0
6,ABE,0.0
7,ABE,0.0
8,ABE,0.0
9,ABE,0.0


**Obtendo taxa de atraso no aeroporto de destino na última hora**

In [50]:
df = df.sort_values(['DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE_DATETIME']).reset_index(drop=True)

df['DESTINATION_AIRPORT_DELAY_MOMENTUM'] = (
    df.groupby('DESTINATION_AIRPORT')
    .rolling('1h', on='SCHEDULED_DEPARTURE_DATETIME', closed='left')['IS_DELAYED']
    .mean()
    .reset_index(level=0, drop=True)
    .fillna(0).values
)

df[['DESTINATION_AIRPORT', 'DESTINATION_AIRPORT_DELAY_MOMENTUM']].head(20)

,DESTINATION_AIRPORT,DESTINATION_AIRPORT_DELAY_MOMENTUM
0,ABE,0.0
1,ABE,0.0
2,ABE,0.0
3,ABE,0.0
4,ABE,0.0
5,ABE,0.0
6,ABE,0.0
7,ABE,0.0
8,ABE,0.0
9,ABE,0.0


**Obtendo volume de voos do aeroporto de origem na mesma janela**

Esta variável atua como um indicador de saturação da infraestrutura aeroportuária ao quantificar a densidade de operações simultâneas em uma janela móvel de uma hora. Essa métrica permite que o modelo diferencie atrasos operacionais isolados de gargalos sistêmicos gerados por picos de demanda.

In [51]:
df = df.sort_values(by=['ORIGIN_AIRPORT', 'SCHEDULED_DEPARTURE_DATETIME'])

df['ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW'] = (
    df.groupby('ORIGIN_AIRPORT')
    .rolling('1h', on='SCHEDULED_DEPARTURE_DATETIME')['DAY']
    .count()
    .reset_index(level=0, drop=True)
    .values - 1
)

df['ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW'] = df['ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW'].fillna(0).astype(int)

df[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW']].head(20)

,ORIGIN_AIRPORT,ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW
1626429,ABE,0
69408,ABE,1
69875,ABE,0
1626637,ABE,0
3649433,ABE,1
70209,ABE,2
70809,ABE,0
1626869,ABE,0
1626938,ABE,0
3650239,ABE,1


**Obtendo perfil do aeroporto de origem**

Para lidar com a alta cardinalidade dos aeroportos de origem, utilizamos aprendizado não supervisionado para consolidar diversas localizações em perfis operacionais distintos, baseados no volume de tráfego e na taxa histórica de atrasos. Esta técnica permite que o modelo capture a sensibilidade da infraestrutura de forma sistêmica, identificando grupos que compartilham padrões semelhantes de instabilidade e congestionamento. Ao substituir identificadores individuais por clusters estatísticos, mitigamos o risco de overfitting e permitimos que o algoritmo generalize o comportamento de aeroportos com perfis de eficiência parecidos, focando na natureza do gargalo operacional em vez de apenas "decorar" nomes de aeroportos específicos. Essa mesma estratégia será aplicada para traçar o perfil de companhias áereas e da rota (aeroporto de origem e destino).

In [ ]:
df_origin_airports_profile = df.groupby('ORIGIN_AIRPORT').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_origin_airports_profile.columns = ['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['ORIGIN_AIRPORT_DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_origin_airports_profile[features_to_scale])

n_clusters = get_best_k(df_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_origin_airports_profile['ORIGIN_AIRPORT_PROFILE'] = kmeans.fit_predict(df_scaled)

In [53]:
plot_scatter(df_origin_airports_profile, 'FLIGHT_VOLUME', 'ORIGIN_AIRPORT_DELAY_RATE', 'ORIGIN_AIRPORT_PROFILE', 'ORIGIN_AIRPORT')

Os aeroportos localizados à extrema direita representam os grandes hubs, que, apesar do altíssimo volume de tráfego, mantêm taxas de atraso moderadas e consistentes. Já no lado esquerdo, onde se concentra a maioria dos aeroportos, o algoritmo diferenciou com sucesso os grupos por performance: o cluster superior (em vermelho) identifica aeroportos de baixo volume mas com alta instabilidade (altos atrasos), enquanto o cluster inferior (em verde) destaca os aeroportos menores e mais pontuais. A interpretação se aplica aos demais perfis abaixo traçados.

**Obtendo perfil da rota**

In [54]:
df['ROUTE'] = df['ORIGIN_AIRPORT'] + '_' + df['DESTINATION_AIRPORT']

df_route_profile = df.groupby('ROUTE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_route_profile.columns = ['ROUTE', 'ROUTE_DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['ROUTE_DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_route_profile[features_to_scale])

n_clusters = get_best_k(df_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_route_profile['ROUTE_PROFILE'] = kmeans.fit_predict(df_scaled)

df_route_profile

,ROUTE,ROUTE_DELAY_RATE,FLIGHT_VOLUME,ROUTE_PROFILE
0,ABE_ATL,0.148984,886,2
1,ABE_DTW,0.179856,695,2
2,ABE_ORD,0.195046,646,2
3,ABI_DFW,0.153294,2231,2
4,ABQ_ATL,0.090113,799,2
...,...,...,...,...
4629,XNA_SFO,0.254902,51,0
4630,XNA_SLC,0.000000,1,2
4631,YAK_CDV,0.129231,325,2
4632,YAK_JNU,0.089231,325,2


In [55]:
plot_scatter(df_route_profile, 'FLIGHT_VOLUME', 'ROUTE_DELAY_RATE', 'ROUTE_PROFILE', 'ROUTE')

**Obtendo perfil da companhia aérea**

In [56]:
df_airline_profile = df.groupby('AIRLINE').agg({
    'IS_DELAYED': ['mean', 'count']
}).reset_index()

df_airline_profile.columns = ['AIRLINE', 'AIRLINE_DELAY_RATE', 'FLIGHT_VOLUME']

scaler = StandardScaler()
features_to_scale = ['AIRLINE_DELAY_RATE', 'FLIGHT_VOLUME']
df_scaled = scaler.fit_transform(df_airline_profile[features_to_scale])

n_clusters = get_best_k(df_scaled)

kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
df_airline_profile['AIRLINE_PROFILE'] = kmeans.fit_predict(df_scaled)

In [57]:
plot_scatter(df_airline_profile, 'FLIGHT_VOLUME', 'AIRLINE_DELAY_RATE', 'AIRLINE_PROFILE', 'AIRLINE')

**Obtendo período do dia**

In [58]:
def get_time_of_day(raw):
    hour = raw // 100
    if 0 <= hour < 6:
        return 'OVERNIGHT'
    elif 6 <= hour < 12:
        return 'MORNING'
    elif 12 <= hour < 18:
        return 'AFTERNOON'
    else:
        return 'EVENING'

df['TIME_OF_DAY'] = df['SCHEDULED_DEPARTURE'].apply(get_time_of_day)

**Obtendo estação do ano**

In [59]:
seasons = {
    12: 'SUMMER', 1: 'SUMMER', 2: 'SUMMER',
    3: 'AUTUMN', 4: 'AUTUMN', 5: 'AUTUMN',
    6: 'WINTER', 7: 'WINTER', 8: 'WINTER',
    9: 'SPRING', 10: 'SPRING', 11: 'SPRING'
}

df['SEASON'] = df['MONTH'].map(seasons)

**Obtendo finais de semana, perfis de distância e feriados**

In [60]:
df['IS_WEEKEND'] = df['DAY_OF_WEEK'].isin([6, 7]).astype(int)

df['HAUL_TYPE'] = pd.qcut(df['DISTANCE'], q=3, labels=['SHORT', 'MEDIUM', 'LONG'])

us_holidays = holidays.US(years=2015)
dates = pd.to_datetime(df['MONTH'].astype(str) + '-' + df['DAY'].astype(str) + '-2015')
df['IS_HOLIDAY'] = dates.dt.date.isin(us_holidays).astype(int)

**Obtendo hora programada de saída e chegada do voo**

In [61]:
df['SCHEDULED_DEPARTURE_HOUR'] = df['SCHEDULED_DEPARTURE'] // 100
df['SCHEDULED_ARRIVAL_HOUR'] = df['SCHEDULED_ARRIVAL'] // 100

**Cruzando dados**

In [62]:
df = df.merge(df_origin_airports_profile[['ORIGIN_AIRPORT', 'ORIGIN_AIRPORT_DELAY_RATE', 'ORIGIN_AIRPORT_PROFILE']], on='ORIGIN_AIRPORT', how='left')
df = df.merge(df_route_profile[['ROUTE', 'ROUTE_DELAY_RATE', 'ROUTE_PROFILE']], on='ROUTE', how='left')
df = df.merge(df_airline_profile[['AIRLINE', 'AIRLINE_DELAY_RATE', 'AIRLINE_PROFILE']], on='AIRLINE', how='left')

**Obtendo correlações**

In [63]:
df_corr = df.copy()

seasons = [['WINTER', 'SPRING', 'AUTUMN', 'SUMMER']]
times_of_day = [['OVERNIGHT', 'MORNING', 'AFTERNOON', 'EVENING']]
hauls_type = [['SHORT', 'MEDIUM', 'LONG']]

encoded_seasons = OrdinalEncoder(categories=seasons)
encoded_times_of_day = OrdinalEncoder(categories=times_of_day)
encoded_hauls_type = OrdinalEncoder(categories=hauls_type)

df_corr['SEASON_NUM'] = encoded_seasons.fit_transform(df_corr[['SEASON']])
df_corr['TIME_OF_DAY_NUM'] = encoded_times_of_day.fit_transform(df_corr[['TIME_OF_DAY']])
df_corr['HAUL_TYPE_NUM'] = encoded_hauls_type.fit_transform(df_corr[['HAUL_TYPE']])

num_cols = [
    'MONTH', 'DAY_OF_WEEK', 'SCHEDULED_DEPARTURE_HOUR', 'SCHEDULED_ARRIVAL_HOUR', 'SCHEDULED_TIME',
    'TIME_OF_DAY_NUM', 'SEASON_NUM', 'DISTANCE', 'ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW', 'ORIGIN_AIRPORT_DELAY_MOMENTUM',
    'DESTINATION_AIRPORT_DELAY_MOMENTUM', 'ORIGIN_AIRPORT_PROFILE', 'ROUTE_PROFILE', 
    'HAUL_TYPE_NUM', 'IS_WEEKEND', 'IS_HOLIDAY', 'AIRPLANE_WAS_DELAYED', 'IS_DELAYED'
]

df_corr = df_corr[num_cols].corr()

plot_correlation_matrix(df_corr)

A matriz de correlação ratifica matematicamente as hipóteses levantadas nas etapas anteriores: o coeficiente de 0,44 entre o atraso da aeronave no voo anterior e a variável target apresenta-se como o preditor mais robusto, confirmando que o estado prévio da aeronave é o principal gatilho para o efeito cascata na malha aérea. Adicionalmente, as correlações de 0,28 e 0,24 para os indicadores de momentum dos aeroportos de origem e destino na última hora validam a influência direta do congestionamento sistêmico em ambas as pontas do fluxo. Enquanto a volatilidade na origem atua no momento exato da decolagem, a degradação no nó de destino sinaliza restrições físicas de recepção de malha. Nota-se também que variáveis de calendário, como feriados e fins de semana, possuem correlação baixa, indicando que a eficiência operacional e a logística imediata sobrepõem-se à sazonalidade simples.

**Selecionando features**

In [64]:
cols = [
    'IS_DELAYED',
    'AIRPLANE_WAS_DELAYED',
    'ORIGIN_AIRPORT_DELAY_MOMENTUM',
    'DESTINATION_AIRPORT_DELAY_MOMENTUM',
    'ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW',
    'HAUL_TYPE',
    'MONTH',
    'SCHEDULED_DEPARTURE_HOUR',
    'IS_WEEKEND',
    'IS_HOLIDAY'
]
df = df[cols]

A seleção das variáveis preditoras fundamenta-se na otimização do espaço de estados e na eliminação de redundâncias estatísticas explicitadas na matriz: o núcleo de modelagem apoia-se nos coeficientes de maior correlação direta com o alvo, consolidando o efeito cascata da aeronave (AIRPLANE_WAS_DELAYED com 0,44) e os sensores bidirecionais de saturação da malha operacional (ORIGIN_AIRPORT_DELAY_MOMENTUM com 0,28 e DESTINATION_AIRPORT_DELAY_MOMENTUM com 0,24). 

Para mitigar os efeitos da multicolinearidade severa que inflaria a variância do modelo, removeu-se os blocos redundantes de alta intensidade: preservou-se apenas o SCHEDULED_DEPARTURE_HOUR como indexador contínuo de acúmulo de atraso ao longo do dia, isolando o acoplamento de 0,94 e 0,70 observado em relação a TIME_OF_DAY e SCHEDULED_ARRIVAL_HOUR. 

De forma análoga, a escolha do HAUL_TYPE atua como uma simplificação estrutural que neutraliza a correlação quase perfeita de 0,98 entre DISTANCE e SCHEDULED_TIME. Por fim, os atributos de volume e sazonalidade macro (ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW, MONTH, IS_WEEKEND e IS_HOLIDAY) foram mantidos; embora apresentem correlação linear próxima a zero, sua inclusão fornece a métrica de densidade de tráfego e contexto temporal necessária para que algoritmos baseados em árvores explorem com precisão as interações não-lineares latentes entre a extensão do voo, os períodos de alta temporada e o colapso físico da malha aérea.

**Balanceando target**

In [65]:
target = 'IS_DELAYED'

df_atrasados = df[df[target] == 1]
df_pontuais = df[df[target] == 0]

n_atrasados = len(df_atrasados)
df_pontuais_bal = df_pontuais.sample(n=n_atrasados, random_state=42)

df_balanced = pd.concat([df_atrasados, df_pontuais_bal])
df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

**Normalizando features**

In [66]:
cols_categoricas = ['HAUL_TYPE']
cols_numericas = [
    'AIRPLANE_WAS_DELAYED',
    'ORIGIN_AIRPORT_DELAY_MOMENTUM',
    'DESTINATION_AIRPORT_DELAY_MOMENTUM',
    'ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW',
    'MONTH',
    'SCHEDULED_DEPARTURE_HOUR',
    'IS_WEEKEND',
    'IS_HOLIDAY'
]

df_normalized = df_balanced.copy()

scaler = StandardScaler()
df_normalized[cols_numericas] = scaler.fit_transform(df_normalized[cols_numericas])

df_normalized = pd.get_dummies(df_normalized, columns=cols_categoricas, dtype=int)

df_normalized.head()

,IS_DELAYED,AIRPLANE_WAS_DELAYED,ORIGIN_AIRPORT_DELAY_MOMENTUM,DESTINATION_AIRPORT_DELAY_MOMENTUM,ORIGIN_AIRPORT_FLIGHTS_ON_THE_SAME_WINDOW,MONTH,SCHEDULED_DEPARTURE_HOUR,IS_WEEKEND,IS_HOLIDAY,HAUL_TYPE_SHORT,HAUL_TYPE_MEDIUM,HAUL_TYPE_LONG
0,1,-0.547400,-0.898713,-0.904391,-0.887294,-0.338099,-1.588269,1.716522,-0.161727,0,0,1
1,1,1.826816,-0.898713,0.719494,-1.046324,-1.521193,-0.749592,-0.582573,-0.161727,0,1,0
2,1,-0.547400,0.490011,0.351049,-0.887294,1.732315,-1.168931,-0.582573,-0.161727,1,0,0
3,0,-0.547400,0.490011,0.507980,-0.569234,0.253448,0.508425,-0.582573,-0.161727,0,1,0
4,0,-0.547400,-0.898713,-0.464987,-0.410204,0.844994,-2.846286,-0.582573,-0.161727,0,0,1


**Exportando features**

In [67]:
df_normalized.to_pickle('../data/curated/features.pkl')